# H₂ Potential Energy Surface Scan: VQE (UCCSD) vs. PySCF FCI & RHF

This notebook runs a 13-point bond-length dissociation scan ($0.30\text{ \AA}$ to $2.50\text{ \AA}$) for $\text{H}_2$ in the minimal `STO-3G` basis:
- **Fermion-to-Qubit Mapping:** Parity mapping with two-qubit $\mathbb{Z}_2$ symmetry reduction (`2` qubits).
- **Ansatz & Optimizer:** `UCCSD` (3 excitation parameters, depth 15, 21 primitive gates) optimized with `SLSQP` (`seed=42`).
- **Warm-Starting:** Each geometry $R_k$ initializes its UCCSD parameters at the converged angles $\vec{\theta}^*_{k-1}$ of the preceding bond length.
- **Classical Baselines:** Exact Restricted Hartree–Fock (`RHF`) and Full Configuration Interaction (`FCI`) computed live via `PySCF`.
- **Export:** Saves the publication figure to `docs/h2_curve.png`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# Ensure repository root is on sys.path when running from notebooks/
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from qmed.vqe_runner import DEFAULT_H2_SCAN_DISTANCES, run_h2_bond_scan, save_h2_curve_plot

In [ ]:
points = run_h2_bond_scan(bond_lengths=DEFAULT_H2_SCAN_DISTANCES, seed=42, warm_start=True)

df_scan = pd.DataFrame([
    {
        "R (Å)": pt.bond_length_angstrom,
        "RHF (Ha)": round(pt.hf_energy, 8),
        "FCI (Ha)": round(pt.fci_energy, 8),
        "VQE (Ha)": round(pt.vqe_energy, 8),
        "|VQE - FCI| (mHa)": round(pt.abs_error_mha, 6),
        "Iterations": pt.iterations,
        "Converged": pt.converged,
        "Warm-Started": pt.warm_started,
    }
    for pt in points
])
df_scan

In [ ]:
output_path = REPO_ROOT / "docs" / "h2_curve.png"
saved_fig_path = save_h2_curve_plot(points, output_path=output_path)
print(f"Saved H2 potential energy surface plot to: {saved_fig_path}")